# 1.9. Analyze metric absolute response to cumulative image degradation

This notebook:
1. Selects one representative fluorescence reference image from a reproducible biological context.
2. Applies six image degradations cumulatively and measures seven full-reference image quality metrics after every iteration.
3. Writes the experiment configuration, TIFF snapshots, and raw and normalized metric tables for visualization in notebook 1.10.

In [1]:
from pathlib import Path

import lance
import pandas as pd

from utils.encoding import decode_pixel_record
from utils.metric_absolute_response import (
    AbsoluteResponseConfig,
    run_iterative_degradation_analysis,
)
from utils.metric_normalization import normalize_metric_results
from utils.validate_config import (
    load_yaml_config,
    require_config_directory,
    require_config_membership,
    require_config_value,
)

/home/weishanli/anaconda3/envs/vs-metric-analysis/lib/python3.12/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.1'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


## Paths and analysis inputs

Resolve the reference-image dataset and metadata generated by notebooks 1.0-1.2. Analysis artifacts are written under the configured analysis directory so visualization does not repeat the metric computation.

In [2]:
config = load_yaml_config("degradation_config.yaml")
analysis_dir = require_config_directory(config, "analysis_out_dir")

metadata_dir = Path("metadata")
platemap_dir = metadata_dir / "platemaps"
barcode_file = metadata_dir / "Barcode_platemap_pilot_data.csv"
reference_lance_dir = analysis_dir / "patches" / "reference_records" / "data.lance"

for required_path in (platemap_dir, barcode_file, reference_lance_dir):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required input does not exist: {required_path}. Run notebooks 1.0-1.2 first."
        )

channels = require_config_value(config, "channels")
input_channel = require_config_membership(config, "input_channel", "channels")
target_channels = [channel for channel in channels if channel != input_channel]
reference_channel = target_channels[0]

# for larger intermediate results
analysis_output_dir = analysis_dir / "metric_absolute_response_analysis" / reference_channel

# for final results 
results_output_dir = Path("results") / "metric_absolute_response_analysis"

## Select a representative reference image

Use the same fixed biological context and random seed on every run, then select the first available site for that plate and well. The first configured fluorescence target channel is the reference channel.

In [3]:
barcode_df = pd.read_csv(barcode_file).rename(columns={"barcode": "Metadata_Plate"})
barcode_df = barcode_df.loc[barcode_df["time_point"] == 24]
platemap_df = pd.concat(
    [
        pd.read_csv(platemap_dir / f"{platemap}.csv").assign(platemap_file=platemap)
        for platemap in barcode_df["platemap_file"].unique()
    ],
    ignore_index=True,
)
barcode_platemap_df = barcode_df.merge(
    platemap_df,
    on="platemap_file",
    how="inner",
    validate="one_to_many",
).rename(columns={"well": "Metadata_Well"})

reference_sample = (
    barcode_platemap_df.loc[
        (barcode_platemap_df["cell_line"] == "U2-OS")
        & (barcode_platemap_df["seeding_density"] == 4000)
        & (barcode_platemap_df["platemap_file"] == "Assay_Plate1_platemap")
    ]
    .sort_values(["Metadata_Plate", "Metadata_Well"])
    .sample(n=1, random_state=42)
)
reference_sample

,Metadata_Plate,time_point,platemap_file,cell_line,row,column,Metadata_Well,seeding_density,condition
190,BR00143976,24,Assay_Plate1_platemap,U2-OS,M,18,M18,4000,standard


In [4]:
reference_dataset = lance.dataset(reference_lance_dir)
sample_plate = reference_sample.iloc[0]["Metadata_Plate"]
sample_well = reference_sample.iloc[0]["Metadata_Well"]

site_metadata = reference_dataset.to_table(
    columns=["Metadata_Plate", "Metadata_Well", "Metadata_Site"],
 ).to_pandas()
matching_sites = site_metadata.loc[
    (site_metadata["Metadata_Plate"] == sample_plate)
    & (site_metadata["Metadata_Well"] == sample_well),
    "Metadata_Site",
 ]
if matching_sites.empty:
    raise ValueError(f"No reference site found for {sample_plate}/{sample_well}.")
sample_site = int(matching_sites.iloc[0])

reference_records = reference_dataset.to_table(
    filter=(
        f"Metadata_Plate = '{sample_plate}' "
        f"AND Metadata_Well = '{sample_well}' "
        f"AND Metadata_Site = {sample_site} "
        f"AND channel = '{reference_channel}'"
    )
 )
if not len(reference_records):
    raise ValueError(
        f"No {reference_channel} reference image found for "
        f"{sample_plate}/{sample_well}/site {sample_site}."
    )

reference_image, reference_metadata = decode_pixel_record(reference_records.to_pylist()[0])
pd.Series(
    {
        "reference_channel": reference_channel,
        "plate": sample_plate,
        "well": sample_well,
        "site": sample_site,
        "image_shape": reference_image.shape,
    },
    name="reference selection",
 )

reference_channel        OrigER
plate                BR00143976
well                        M18
site                          1
image_shape          (256, 256)
Name: reference selection, dtype: object

## Run metric absolute response analysis

Each transform is applied to its own evolving image for 500 cumulative iterations. The utility evaluates every metric after each step and saves a TIFF snapshot every 10 iterations.

In [5]:
response_config = AbsoluteResponseConfig(
    iterations=500,
    snapshot_interval=10,
    noise_fraction=0.01,
)

metric_results = run_iterative_degradation_analysis(
    reference_image,
    reference_channel,
    analysis_output_dir,
    config=response_config,
)
metric_results

Completed gauss_noise: 500 iterations
Completed gaussian_blur: 500 iterations
Completed grid_distortion: 500 iterations
Completed random_gamma: 500 iterations
Completed erode: 500 iterations
Completed dilate: 500 iterations


,reference_channel,transform_name,iteration,metric_name,metric_value
0,OrigER,gauss_noise,1,ssim,0.983872
1,OrigER,gauss_noise,1,psnr,48.245155
2,OrigER,gauss_noise,1,mae,0.003130
3,OrigER,gauss_noise,1,lpips,0.045597
4,OrigER,gauss_noise,1,dists,0.151885
...,...,...,...,...,...
20995,OrigER,dilate,500,mae,0.382934
20996,OrigER,dilate,500,lpips,0.780357
20997,OrigER,dilate,500,dists,0.686365
20998,OrigER,dilate,500,foreground_ssim,0.370244


In [6]:
normalized_metric_results = normalize_metric_results(metric_results)
normalized_metric_results.to_parquet(
    results_output_dir / "normalized_metric_results.parquet",
    index=False,
    compression="zstd",
)

normalized_metric_results.groupby(
    ["transform_name", "metric_name"],
    observed=True,
)["normalized_metric_value"].agg(["min", "max"])

min       max
transform_name  metric_name                        
dilate          dists            0.313504  0.954729
                foreground_psnr  0.212940  0.727157
                foreground_ssim  0.368136  0.971936
                lpips            0.204284  0.988304
                mae              0.617066  0.996342
                psnr             0.165315  0.847186
                ssim             0.102606  0.986063
erode           dists            0.385227  0.961925
                foreground_psnr  0.352876  0.720898
                foreground_ssim  0.054175  0.970032
                lpips            0.560573  0.984631
                mae              0.973260  0.996563
                psnr             0.500095  0.852406
                ssim             0.700408  0.986342
gauss_noise     dists            0.480494  0.848115
                foreground_psnr  0.439092  0.954060
                foreground_ssim  0.206775  0.988958
                lpips            0.049355  0.954403
                mae              0.935871  0.996870
                psnr             0.431600  0.964903
                ssim             0.091057  0.983872
gaussian_blur   dists            0.515697  0.970950
                foreground_psnr  0.456368  0.978276
                foreground_ssim  0.657264  0.994474
                lpips            0.588544  0.983426
                mae              0.978967  0.999278
                psnr             0.586438  1.000000
                ssim             0.709770  0.998590
grid_distortion dists            0.504698  0.989558
                foreground_psnr  0.363334  0.799071
                foreground_ssim  0.139232  0.975122
                lpips            0.517841  0.995089
                mae              0.953007  0.998057
                psnr             0.455314  0.925098
                ssim             0.491359  0.991846
random_gamma    dists            0.371701  0.997007
                foreground_psnr  0.348897  0.883475
                foreground_ssim  0.007469  0.998319
                lpips            0.560251  0.999186
                mae              0.969964  0.998118
                psnr             0.495096  1.000000
                ssim             0.539756  0.997248

## Outputs

Notebook 1.10 reads `normalized_metric_results.parquet` and per-transform visualize montages.